# 3. Overlap Analysis

Analyzes the overlap between bias circuits found for different bias types
(gender vs demographic). Creates blue matrices showing shared edges,
similar to the overlap analysis in the MI_Bias paper.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

sns.set_theme(style='white')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

## Load Both Edge Sets

In [ ]:
gender_path = '../results/top_edges_gender.json'
demo_path = '../results/top_edges_demographic.json'

if not os.path.exists(gender_path) or not os.path.exists(demo_path):
    print('ERROR: Both edge files must exist.')
    print('Run script 02 for BOTH datasets first:')
    print('  python scripts/02_find_circuits.py --dataset data/gender_bias.json')
    print('  python scripts/02_find_circuits.py --dataset data/demographic_bias.json')
else:
    with open(gender_path, 'r') as f:
        gender_edges = json.load(f)
    with open(demo_path, 'r') as f:
        demo_edges = json.load(f)
    
    print(f'Gender bias edges:      {len(gender_edges)}')
    print(f'Demographic bias edges: {len(demo_edges)}')

## Compute Edge Overlap

In [ ]:
def edge_key(e):
    """Create a unique key for an edge."""
    return (e['src_layer'], e['src_type'], e['src_head'],
            e['dst_layer'], e['dst_type'], e['dst_head'])

gender_set = set(edge_key(e) for e in gender_edges)
demo_set = set(edge_key(e) for e in demo_edges)

overlap = gender_set & demo_set
gender_only = gender_set - demo_set
demo_only = demo_set - gender_set

print(f'Shared edges:           {len(overlap)}')
print(f'Gender-only edges:      {len(gender_only)}')
print(f'Demographic-only edges: {len(demo_only)}')
print(f'Jaccard similarity:     {len(overlap) / len(gender_set | demo_set):.3f}')

## Overlap Venn Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Simple Venn-like visualization
from matplotlib.patches import Circle

c1 = Circle((0.35, 0.5), 0.3, alpha=0.4, color='#4A90D9', label='Gender Bias')
c2 = Circle((0.65, 0.5), 0.3, alpha=0.4, color='#2ECC71', label='Demographic Bias')
ax.add_patch(c1)
ax.add_patch(c2)

ax.text(0.25, 0.5, f'{len(gender_only)}', ha='center', va='center',
        fontsize=20, fontweight='bold', color='#2C5F8A')
ax.text(0.5, 0.5, f'{len(overlap)}', ha='center', va='center',
        fontsize=20, fontweight='bold', color='#1a1a1a')
ax.text(0.75, 0.5, f'{len(demo_only)}', ha='center', va='center',
        fontsize=20, fontweight='bold', color='#1E8449')

ax.text(0.25, 0.15, 'Gender\nOnly', ha='center', va='center', fontsize=11, color='#2C5F8A')
ax.text(0.5, 0.15, 'Shared', ha='center', va='center', fontsize=11, color='#333')
ax.text(0.75, 0.15, 'Demographic\nOnly', ha='center', va='center', fontsize=11, color='#1E8449')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Edge Overlap Between Bias Types', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/edge_overlap_venn.png', dpi=200, bbox_inches='tight')
plt.show()

## Layer-Level Overlap Matrix (Blue Heatmap)

In [ ]:
n_layers = max(
    max(e['src_layer'] for e in gender_edges + demo_edges),
    max(e['dst_layer'] for e in gender_edges + demo_edges)
) + 1

# Build overlap matrix: overlap_matrix[src_layer, dst_layer] = number of shared edges
# from that source layer to that destination layer
overlap_matrix = np.zeros((n_layers, n_layers))
gender_matrix = np.zeros((n_layers, n_layers))
demo_matrix = np.zeros((n_layers, n_layers))

for e in gender_edges:
    gender_matrix[e['src_layer'], e['dst_layer']] += e['score']

for e in demo_edges:
    demo_matrix[e['src_layer'], e['dst_layer']] += e['score']

# Overlap = element-wise min of normalized scores
if gender_matrix.max() > 0:
    gender_norm = gender_matrix / gender_matrix.max()
else:
    gender_norm = gender_matrix

if demo_matrix.max() > 0:
    demo_norm = demo_matrix / demo_matrix.max()
else:
    demo_norm = demo_matrix

overlap_matrix = np.minimum(gender_norm, demo_norm)

# Blue colormap
blue_cmap = LinearSegmentedColormap.from_list('blue_overlap', [
    '#f0f4ff',
    '#b3d4ff',
    '#4A90D9',
    '#1a5276',
    '#0a1f3d',
])

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Gender circuit matrix
im1 = axes[0].imshow(gender_norm, cmap='Purples', aspect='auto', interpolation='nearest')
axes[0].set_title('Gender Bias Circuit', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Destination Layer')
axes[0].set_ylabel('Source Layer')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Demographic circuit matrix
im2 = axes[1].imshow(demo_norm, cmap='Greens', aspect='auto', interpolation='nearest')
axes[1].set_title('Demographic Bias Circuit', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Destination Layer')
axes[1].set_ylabel('Source Layer')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

# Overlap matrix (blue)
im3 = axes[2].imshow(overlap_matrix, cmap=blue_cmap, aspect='auto', interpolation='nearest')
axes[2].set_title('Shared Bias Circuit (Overlap)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Destination Layer')
axes[2].set_ylabel('Source Layer')
plt.colorbar(im3, ax=axes[2], shrink=0.8)

for ax in axes:
    ax.set_xticks(range(n_layers))
    ax.set_yticks(range(n_layers))

plt.suptitle('Bias Circuit Overlap Analysis (Layer × Layer)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/circuit_overlap_matrices.png', dpi=200, bbox_inches='tight')
plt.show()

## Head-Level Overlap (Shared Attention Heads)

In [ ]:
# Find which specific heads are shared as sources
def get_src_heads(edges_list):
    heads = set()
    for e in edges_list:
        if e['src_type'] == 'attn' and e['src_head'] is not None:
            heads.add((e['src_layer'], e['src_head']))
    return heads

gender_heads = get_src_heads(gender_edges)
demo_heads = get_src_heads(demo_edges)
shared_heads = gender_heads & demo_heads

print(f'Gender source heads: {len(gender_heads)}')
print(f'Demographic source heads: {len(demo_heads)}')
print(f'Shared source heads: {len(shared_heads)}')
print(f'\nShared heads: {sorted(shared_heads)}')

In [ ]:
# Visualize shared heads on a grid
n_heads_max = max(
    max((h for _, h in gender_heads), default=0),
    max((h for _, h in demo_heads), default=0)
) + 1

head_grid = np.zeros((n_layers, n_heads_max))
# 1 = gender only, 2 = demo only, 3 = shared
for l, h in gender_heads:
    head_grid[l, h] = 1
for l, h in demo_heads:
    if head_grid[l, h] == 1:
        head_grid[l, h] = 3  # shared
    else:
        head_grid[l, h] = 2

# Custom colormap: white, blue (gender), green (demo), red (shared)
from matplotlib.colors import ListedColormap
overlap_cmap = ListedColormap(['#f5f5f5', '#4A90D9', '#2ECC71', '#E74C3C'])

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(head_grid, cmap=overlap_cmap, aspect='auto', interpolation='nearest',
               vmin=0, vmax=3)

ax.set_xlabel('Head Index', fontsize=13)
ax.set_ylabel('Layer', fontsize=13)
ax.set_title('Attention Head Involvement in Bias Circuits', fontsize=15, fontweight='bold')
ax.set_xticks(range(n_heads_max))
ax.set_yticks(range(n_layers))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#f5f5f5', edgecolor='gray', label='Not involved'),
    Patch(facecolor='#4A90D9', label='Gender bias only'),
    Patch(facecolor='#2ECC71', label='Demographic bias only'),
    Patch(facecolor='#E74C3C', label='Shared (both)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/head_overlap_grid.png', dpi=200, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
print('=' * 50)
print('  CIRCUIT OVERLAP SUMMARY')
print('=' * 50)
print(f'  Gender bias edges:       {len(gender_edges)}')
print(f'  Demographic bias edges:  {len(demo_edges)}')
print(f'  Shared edges:            {len(overlap)}')
print(f'  Jaccard similarity:      {len(overlap) / len(gender_set | demo_set):.3f}')
print(f'  Shared attention heads:  {len(shared_heads)}')
print(f'  Gender-only heads:       {len(gender_heads - demo_heads)}')
print(f'  Demographic-only heads:  {len(demo_heads - gender_heads)}')
print('=' * 50)